# IOAI — 2026 Mock Chemical Kinetics (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-chemical-kinetics/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Chemical Kinetics — 모범답안 (물리 통찰 특징 + 랜덤포레스트)

반감기는 리간드 L 이 반응을 억제(c(L)↑ → t1/2↑)하고 D 가 촉진하는 구조다. 원 특징(c(L2M),c(D),c(L))에
**비율 특징**(L/D, L/L2M)과 곱 특징을 더해 랜덤포레스트로 회귀한다. 점수 ≈ 0.80 (평균예측 0.21 대비).

In [ ]:
import pandas as pd, numpy as np
train = pd.read_csv("data/train.csv")            # exp, L2M0, D0, L0, T12
test  = pd.read_csv("data/test.csv")             # exp, L2M0, D0, L0
print(train.shape, "| T12 range", round(train.T12.min(),1), "-", round(train.T12.max(),1))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
F = ["L2M0","D0","L0"]
def feats(df):
    x = df[F].copy()
    x["L_D"] = df.L0 / df.D0.clip(lower=1e-6)
    x["L_L2M"] = df.L0 / df.L2M0.clip(lower=1e-6)
    x["prod"] = df.L0 * df.D0
    return x
rf = RandomForestRegressor(n_estimators=400, random_state=0).fit(feats(train), train.T12)
pred = rf.predict(feats(test))
pd.DataFrame({"Experiment Number": test.exp, "t12": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(test))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)